# SQuAD v1.1 — QA trích xuất span — bản tối ưu

Primary Exact Match ↑; thêm token F1 ↑. BiDAF-style với char và contextual attention, scratch.

Mô hình nhiệm vụ, embedding và tokenizer đều học từ train. Người dùng đã cho phép pretrained **riêng trong metric** COMET/BERTScore/ERRANT. Không dùng evaluator để tạo dự đoán hay cập nhật mô hình. Chạy độc lập bằng Restart Kernel → Run All; không cần notebook khác/fewshot/workbook. Chưa huấn luyện đầy đủ và không kèm điểm benchmark.

**Cách viết training/checkpoint:** training loop được viết trực tiếp trong cell (sklearn dùng fit); lưu gọn khi validation cải thiện. Checkpoint phục vụ inference, không lưu optimizer/scheduler/RNG để resume.


## 1. Chuẩn bị thư viện

PyTorch là framework huấn luyện. TFDS chỉ reader CPU, dùng Python3.11/3.12 khi cần TensorFlow. Có nhánh original_json từ chính SQuAD để đọc trên host không có TF.


In [ ]:
import importlib.util
import subprocess
import sys
REQUIRED_PACKAGES = []
missing = [package for module,package in REQUIRED_PACKAGES if importlib.util.find_spec(module) is None]
if missing: subprocess.check_call([sys.executable,"-m","pip","install",*missing])
if importlib.util.find_spec("torch") is None:
    raise RuntimeError("Cần kernel có PyTorch CPU/CUDA phù hợp. Cài PyTorch rồi Restart Kernel → Run All.")
from pathlib import Path
from collections import Counter,defaultdict
from types import SimpleNamespace
import gc
import hashlib
import io
import json
import math
import random
import re
import string
import unicodedata
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset,DataLoader
from torch.nn.utils.rnn import pack_padded_sequence,pad_packed_sequence
from tqdm.auto import tqdm


## 2. Cấu hình cố định

Đây là extractive QA, không ép thành seq2seq. Không dùng BERT/tokenizer/embedding pretrained. Chọn cuda/cpu đơn giản, không AMP/auto batch.


In [ ]:
SEED = 42
VARIANT = "optimized"
ARTIFACT_DIR = Path("artifacts/qa_squad_v11") / VARIANT
CACHE_DIR = ARTIFACT_DIR / "data_cache"
METRIC_CACHE_DIR = ARTIFACT_DIR / "metric_cache"
CHECKPOINT_PATH = ARTIFACT_DIR / "best_model.pt"
TASK_PRETRAINING_POLICY = "scratch"
TASK_PRETRAINED_COMPONENTS = []
ALLOW_PRETRAINED_EVALUATORS = True  # Ngoại lệ người dùng cho phép chỉ dùng để chấm metric.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
ARTIFACT_DIR.mkdir(parents=True,exist_ok=True)
print(f"Thiết bị: {DEVICE}; biến thể: {VARIANT}; PyTorch {torch.__version__}; Python {sys.version.split()[0]}")


SOURCE = {'row': 3, 'task': 'Extractive QA', 'dataset': 'SQuAD v1.1', 'tfds_builder': 'squad/v1.1:3.0.0', 'original_hint': 'squad/v1.1.0 (đã sửa config thành v1.1, builder version3.0.0)', 'url': 'https://www.tensorflow.org/datasets/catalog/squad'}
DATA_MODE = "tfds"
SOURCE["data_mode"] = DATA_MODE
LOCAL_DATA_DIR = Path("data/qa_squad_v11")
EVALUATION_COMPONENTS = []
VAL_FRACTION = 0.10
VOCAB_SIZE = 40000
MAX_CONTEXT_TOKENS = 384
WINDOW_STEP = 256
MAX_QUESTION_TOKENS = 64
MAX_ANSWER_TOKENS = 64
MAX_WORD_CHARS = 32
NEGATIVE_WINDOW_WEIGHT = 0.5
HIDDEN_DIM = 128
DROPOUT = 0.2
BATCH_SIZE = 24
EVAL_BATCH_SIZE = 24
EPOCHS = 20
PATIENCE = 5
LEARNING_RATE = 0.001
WARMUP_STEPS = 0
WEIGHT_DECAY = 0.0001
GRAD_CLIP = 1.0
PRIMARY_METRIC = "exact_match"


In [ ]:
def fingerprint(value):
    text=json.dumps(value,ensure_ascii=False,sort_keys=True,separators=(",",":"))
    return hashlib.sha256(text.encode()).hexdigest()

def clean_text(text):
    return " ".join(unicodedata.normalize("NFC",text).split())

def read_jsonl(path):
    if not path.is_file():raise FileNotFoundError(f"Thiếu {path}; cần JSONL xuất từ đúng nguồn/schema đã ghi.")
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

def tfds_rows(builder,splits,supervised=False):
    # TensorFlow chỉ là reader CPU; PyTorch vẫn sử dụng DEVICE CUDA khi có.
    if importlib.util.find_spec("tensorflow") is None:
        if sys.version_info >= (3,13):
            raise RuntimeError("TFDS reader cần TensorFlow. Host Python này chưa có wheel phù hợp. "
                "Dùng kernel Python3.11/3.12 + tensorflow==2.18.1 tensorflow-datasets==4.9.10, "
                "hoặc chọn DATA_MODE='local_jsonl' với dữ liệu xuất đúng nguồn. Không có dữ liệu thay thế tự động.")
        subprocess.check_call([sys.executable,"-m","pip","install","tensorflow==2.18.1","tensorflow-datasets==4.9.10"])
    elif importlib.util.find_spec("tensorflow_datasets") is None:
        subprocess.check_call([sys.executable,"-m","pip","install","tensorflow-datasets==4.9.10"])
    import tensorflow as tf
    try:tf.config.set_visible_devices([],"GPU")
    except RuntimeError as exc:
        raise RuntimeError("TensorFlow đã khởi tạo GPU. Restart Kernel rồi Run All để reader chạy CPU.") from exc
    import tensorflow_datasets as tfds
    try:
        data,info=tfds.load(builder,split=splits,as_supervised=supervised,with_info=True,
                            data_dir=str(CACHE_DIR),shuffle_files=False,try_gcs=True)
        print("TFDS:",builder,"; phiên bản thư viện",tfds.__version__,"; split counts",{k:v.num_examples for k,v in info.splits.items()})
        return {split:list(tfds.as_numpy(dataset)) for split,dataset in zip(splits,data)}
    except Exception as exc:
        raise RuntimeError(f"TFDS không đọc được {builder}. Kiểm tra cache/mạng/nguồn download; "
            "có thể dùng local_jsonl đã xuất từ đúng builder. Xem mục dữ liệu trong đề xuất, không đổi dataset âm thầm. "
            f"Chi tiết: {exc}") from exc

def decode_utf8(value):
    return value.decode("utf-8") if isinstance(value,(bytes,np.bytes_)) else str(value)

def to_device(batch):
    return {k:v.to(DEVICE) if torch.is_tensor(v) and k not in ("lengths","question_lengths") else v for k,v in batch.items()}

def make_loader(dataset,train=False):
    return DataLoader(dataset,batch_size=BATCH_SIZE if train else EVAL_BATCH_SIZE,
                      shuffle=train,num_workers=0,collate_fn=collate_batch,
                      generator=torch.Generator().manual_seed(SEED))


## 3. Dữ liệu, offset và split

Sửa hint thành squad/v1.1:3.0.0; config v1.1, phiên bản builder3.0.0 sửa lỗi khoảng trắng. Chia title Wikipedia trong train thành train/validation; official validation chỉ làm held-out. Context giữ nguyên tuyệt đối để offset đúng.


In [ ]:
def adapt(row):
    answers=row["answers"]
    if isinstance(answers,dict):
        starts=answers["answer_start"];texts=answers["text"]
        answers=[{"answer_start":int(start),"text":decode_utf8(text)} for start,text in zip(starts,texts)]
    else:answers=[{"answer_start":int(a["answer_start"]),"text":decode_utf8(a["text"])} for a in answers]
    record={"id":decode_utf8(row["id"]),"title":decode_utf8(row["title"]),
            "context":decode_utf8(row["context"]),"question":decode_utf8(row["question"]),"answers":answers}
    if not answers:raise ValueError("SQuAD v1.1 phải có answer; không dùng v2 như v1.1.")
    for answer in answers:
        s=answer["answer_start"];e=s+len(answer["text"])
        if s<0 or record["context"][s:e]!=answer["text"]:raise ValueError(f"Offset gold không khớp ở{record['id']}; không strip/normalize context.")
    return record

if DATA_MODE=="tfds":
    raw=tfds_rows(SOURCE["tfds_builder"],["train","validation"])
elif DATA_MODE=="original_json":
    import urllib.request
    raw={}
    CACHE_DIR.mkdir(parents=True,exist_ok=True)
    for split,file in (("train","train-v1.1.json"),("validation","dev-v1.1.json")):
        path=CACHE_DIR/file
        if not path.is_file():
            try:path.write_bytes(urllib.request.urlopen("https://rajpurkar.github.io/SQuAD-explorer/dataset/"+file,timeout=120).read())
            except Exception as exc:raise RuntimeError(f"Không tải được JSON SQuAD gốc. Đặt đúng file tại{path}. {exc}") from exc
        document=json.loads(path.read_text(encoding="utf-8"));rows=[]
        if document["version"]!="1.1":raise ValueError("Không đúng SQuAD1.1.")
        for article in document["data"]:
            for paragraph in article["paragraphs"]:
                for qa in paragraph["qas"]:rows.append({**qa,"title":article["title"],"context":paragraph["context"]})
        raw[split]=rows
elif DATA_MODE=="local_jsonl":raw={split:read_jsonl(LOCAL_DATA_DIR/f"{split}.jsonl") for split in ("train","validation")}
else:raise ValueError("DATA_MODE cần tfds, original_json hoặc local_jsonl.")

pool=sorted([adapt(row) for row in raw["train"]],key=lambda r:r["id"])
test_records=sorted([adapt(row) for row in raw["validation"]],key=lambda r:r["id"])
titles=sorted({r["title"] for r in pool})
random.Random(SEED).shuffle(titles)
val_titles=set(titles[:max(1,math.ceil(VAL_FRACTION*len(titles)))])
train_records=[r for r in pool if r["title"] not in val_titles]
val_records=[r for r in pool if r["title"] in val_titles]
if not train_records or not val_records:raise ValueError("Không đủ Wikipedia titles để chia validation.")
context_sets=[{r["context"] for r in records} for records in (train_records,val_records,test_records)]
if context_sets[0]&context_sets[1] or context_sets[0]&context_sets[2] or context_sets[1]&context_sets[2]:
    raise ValueError("Context trùng giữa split; xử lý nhóm context trước khi học.")
if len({r["id"] for r in train_records+val_records+test_records})!=len(train_records+val_records+test_records):
    raise ValueError("Question ID trùng xuyên split.")
SPLIT_FINGERPRINTS={name:fingerprint(records) for name,records in (("train",train_records),("validation",val_records),("held_out_official_validation",test_records))}
print("Question counts:",len(train_records),len(val_records),len(test_records))
print("Title counts:",len({r["title"] for r in train_records}),len(val_titles),len({r["title"] for r in test_records}))
print("Fingerprint:",SPLIT_FINGERPRINTS)
print("Official validation làm held-out có nhãn; không có điểm original hidden test.")


## 4. Vocabulary, windows và Dataset

Vocabulary chỉ train, word regex có offsets, char feature train-only. Mọi context đi qua toàn bộ windows384/step256, không lấy riêng window chứa answer khi eval. Slot0 CLS chỉ cho window không có answer, final v1.1 luôn chọn span không rỗng. Gold giữa token được báo giới hạn, không sửa gold khi chấm.


In [ ]:
import bisect
PAD_ID,UNK_ID,CLS_ID=0,1,2
TOKEN_PATTERN=r"\w+|[^\w\s]"
_CONTEXT_CACHE={}
def context_tokens(text):
    if text not in _CONTEXT_CACHE:
        matches=list(re.finditer(TOKEN_PATTERN,text))
        _CONTEXT_CACHE[text]={"words":[m.group() for m in matches],"offsets":[m.span() for m in matches]}
    return _CONTEXT_CACHE[text]
def question_words(text):return re.findall(TOKEN_PATTERN,text)[:MAX_QUESTION_TOKENS] or ["<UNK>"]

counts=Counter()
characters=set()
seen_context=set()
for record in train_records:
    if record["context"] not in seen_context:
        seen_context.add(record["context"])
        words=context_tokens(record["context"])["words"]
        counts.update(w.lower() for w in words);characters.update(ch for w in words for ch in w)
    words=re.findall(TOKEN_PATTERN,record["question"])
    counts.update(w.lower() for w in words);characters.update(ch for w in words for ch in w)
word_vocab={"<PAD>":PAD_ID,"<UNK>":UNK_ID,"<CLS>":CLS_ID}
word_vocab.update({word:i+3 for i,word in enumerate(sorted(counts,key=lambda w:(-counts[w],w))[:VOCAB_SIZE-3])})
char_vocab={"<PAD>":0,"<UNK>":1}
char_vocab.update({ch:i+2 for i,ch in enumerate(sorted(characters))})
TOKENIZER_STATE={"word_vocab":word_vocab,"char_vocab":char_vocab,"regex":TOKEN_PATTERN,
                 "pad":PAD_ID,"unk":UNK_ID,"cls":CLS_ID}
PREPROCESS_CONFIG={"max_context_tokens":MAX_CONTEXT_TOKENS,"window_step":WINDOW_STEP,
                    "max_question_tokens":MAX_QUESTION_TOKENS,"max_answer_tokens":MAX_ANSWER_TOKENS,
                    "max_word_chars":MAX_WORD_CHARS,"context_normalization":"không thay đổi chuỗi gốc",
                    "train_alignment":"minimal covering token span","decode":"best joint span minus CLS score across all windows"}
def restore_tokenizer(state):
    global word_vocab,char_vocab
    word_vocab,char_vocab=state["word_vocab"],state["char_vocab"]

def answer_token_span(context,answer):
    offsets=context_tokens(context)["offsets"]
    starts=[s for s,e in offsets];ends=[e for s,e in offsets]
    s=answer["answer_start"];e=s+len(answer["text"])
    left=bisect.bisect_right(ends,s);right=bisect.bisect_left(starts,e)-1
    if left>right or right>=len(offsets):raise ValueError("Answer không giao token nào.")
    exact=offsets[left][0]==s and offsets[right][1]==e
    return left,right,exact

class QADataset(Dataset):
    def __init__(self,records,training=False):
        self.records=records;self.features=[]
        for record in records:
            n=len(context_tokens(record["context"])["words"])
            if not n:raise ValueError("Context không có token.")
            gold=[answer_token_span(record["context"],a)[:2] for a in record["answers"]] if training else []
            has_positive=False
            for start in range(0,n,WINDOW_STEP):
                stop=min(start+MAX_CONTEXT_TOKENS,n);targets=(0,0)
                for a,b in gold:
                    if start<=a<=b<stop:targets=(a-start+1,b-start+1);has_positive=True;break
                self.features.append({"record":record,"start":start,"stop":stop,"targets":targets})
                if stop==n:break
            if training and not has_positive:raise ValueError("Không có window chứa answer; cần tăng context cap/overlap thủ công.")
    def __len__(self):return len(self.features)
    def __getitem__(self,index):
        feature=self.features[index];record=feature["record"]
        cached=context_tokens(record["context"])
        words=["<CLS>"]+cached["words"][feature["start"]:feature["stop"]]
        offsets=[(-1,-1)]+cached["offsets"][feature["start"]:feature["stop"]]
        return {"words":words,"question":question_words(record["question"]),"offsets":offsets,
                "targets":feature["targets"],"record":record,"id":f"{record['id']}:{feature['start']}"}

def encode_words(rows,width):
    words=torch.zeros(len(rows),width,dtype=torch.long)
    chars=torch.zeros(len(rows),width,MAX_WORD_CHARS,dtype=torch.long)
    for i,tokens in enumerate(rows):
        words[i,:len(tokens)]=torch.tensor([CLS_ID if w=="<CLS>" else word_vocab.get(w.lower(),UNK_ID) for w in tokens])
        for j,word in enumerate(tokens):
            ids=[char_vocab.get(ch,1) for ch in word[:MAX_WORD_CHARS]]
            if ids:chars[i,j,:len(ids)]=torch.tensor(ids)
    return words,chars

def collate_batch(items):
    lengths=torch.tensor([len(item["words"]) for item in items])
    question_lengths=torch.tensor([len(item["question"]) for item in items])
    context,context_chars=encode_words([item["words"] for item in items],int(lengths.max()))
    question,question_chars=encode_words([item["question"] for item in items],int(question_lengths.max()))
    return {"context":context,"context_chars":context_chars,"question":question,"question_chars":question_chars,
            "lengths":lengths,"question_lengths":question_lengths,
            "context_mask":torch.arange(context.size(1))[None]<lengths[:,None],
            "question_mask":torch.arange(question.size(1))[None]<question_lengths[:,None],
            "targets":torch.tensor([item["targets"] for item in items]),
            "ids":[item["id"] for item in items],"items":items}

train_loader=make_loader(QADataset(train_records,True),True)
val_loader=make_loader(QADataset(val_records))
test_loader=make_loader(QADataset(test_records))
print("Word/char vocab chỉ train:",len(word_vocab),len(char_vocab))
print("Số windows:",len(train_loader.dataset),len(val_loader.dataset),len(test_loader.dataset))
for name,records in (("train",train_records),("validation",val_records),("held_out",test_records)):
    alignment=[answer_token_span(r["context"],a) for r in records for a in r["answers"]]
    print(name,"gold answer boundaries giữa token:",sum(not exact for a,b,exact in alignment),
          "; gold dài hơn answer cap:",sum(b-a+1>MAX_ANSWER_TOKENS for a,b,exact in alignment))
print("Nhãn train là token span bao phủ tối thiểu; không sửa gold text khi chấm EM/F1. Giới hạn ranh giới được báo rõ.")


## 5. Mô hình

BiDAF-style: word/char-CNN, BiLSTM, attention hai chiều context–question, modeling/end BiLSTM và span heads.


In [ ]:
class QAModel(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.embedding=nn.Embedding(cfg["n_words"],128,padding_idx=0)
        self.char_embedding=nn.Embedding(cfg["n_chars"],16,padding_idx=0)
        self.char_cnn=nn.Conv1d(16,32,3,padding=1)
        self.encoder=nn.LSTM(160,cfg["hidden_dim"],batch_first=True,bidirectional=True)
        dim=2*cfg["hidden_dim"]
        self.sim_context=nn.Linear(dim,1,bias=False)
        self.sim_question=nn.Linear(dim,1,bias=False)
        self.sim_product=nn.Parameter(torch.empty(dim));nn.init.normal_(self.sim_product,std=.02)
        self.modeling=nn.LSTM(4*dim,cfg["hidden_dim"],batch_first=True,bidirectional=True)
        self.end_encoder=nn.LSTM(dim,cfg["hidden_dim"],batch_first=True,bidirectional=True)
        self.start_head=nn.Linear(5*dim,1)
        self.end_head=nn.Linear(5*dim,1)
        self.dropout=nn.Dropout(cfg["dropout"])
    def char_features(self,chars):
        b,n,c=chars.shape;flat=chars.reshape(b*n,c)
        values=self.char_cnn(self.char_embedding(flat).transpose(1,2)).relu()
        values=values.masked_fill((flat==0)[:,None],-10000).max(-1).values
        values=torch.where((flat!=0).any(-1,keepdim=True),values,torch.zeros_like(values))
        return values.reshape(b,n,-1)
    def sequence(self,module,x,lengths):
        packed=pack_padded_sequence(self.dropout(x),lengths,batch_first=True,enforce_sorted=False)
        hidden,_=module(packed)
        return pad_packed_sequence(hidden,batch_first=True,total_length=x.size(1))[0]
    def forward(self,batch):
        c=torch.cat([self.embedding(batch["context"]),self.char_features(batch["context_chars"])],-1)
        q=torch.cat([self.embedding(batch["question"]),self.char_features(batch["question_chars"])],-1)
        c=self.sequence(self.encoder,c,batch["lengths"]);q=self.sequence(self.encoder,q,batch["question_lengths"])
        similarity=self.sim_context(c)+self.sim_question(q).transpose(1,2)+torch.bmm(c*self.sim_product,q.transpose(1,2))
        similarity=similarity.masked_fill(~batch["question_mask"][:,None],-torch.inf)
        attended_q=torch.bmm(similarity.softmax(-1),q)
        c_weights=similarity.max(-1).values.masked_fill(~batch["context_mask"],-torch.inf).softmax(-1)
        attended_c=torch.bmm(c_weights[:,None],c).expand(-1,c.size(1),-1)
        fused=torch.cat([c,attended_q,c*attended_q,c*attended_c],-1)
        modeled=self.sequence(self.modeling,fused,batch["lengths"])
        end_modeled=self.sequence(self.end_encoder,modeled,batch["lengths"])
        start=self.start_head(self.dropout(torch.cat([fused,modeled],-1))).squeeze(-1)
        end=self.end_head(self.dropout(torch.cat([fused,end_modeled],-1))).squeeze(-1)
        return start.masked_fill(~batch["context_mask"],-10000),end.masked_fill(~batch["context_mask"],-10000)
def build_model(cfg):return QAModel(cfg)


In [ ]:
MODEL_CONFIG={"n_words":len(word_vocab),"n_chars":len(char_vocab),"hidden_dim":HIDDEN_DIM,"dropout":DROPOUT}


## 6. Loss, span decoding và metric

Start/end CE có trọng số window âm0.5. Decode joint end≥start, answer≤64tokens, không CLS/PAD; gộp windows bằng margin so với CLS. Chấm một lần/question, EM/token-F1 chuẩn v1.1 lấy tốt nhất trên tất cả references, thang0..100.


In [ ]:
def normalize_answer(text):
    text=text.lower()
    text="".join(ch for ch in text if ch not in string.punctuation)
    text=re.sub(r"\b(a|an|the)\b"," ",text)
    return " ".join(text.split())
def answer_metrics(prediction,references):
    em=[];f1=[]
    for gold in references:
        p=normalize_answer(prediction);g=normalize_answer(gold)
        em.append(float(p==g))
        p_tokens=p.split();g_tokens=g.split();common=sum((Counter(p_tokens)&Counter(g_tokens)).values())
        if not common:f1.append(0.);continue  # Quy tắc evaluator SQuAD v1.1.
        precision=common/len(p_tokens);recall=common/len(g_tokens)
        f1.append(2*precision*recall/(precision+recall))
    return max(em),max(f1)

def compute_loss(model,batch):
    start,end=model(batch);targets=batch["targets"]
    loss=nn.functional.cross_entropy(start,targets[:,0],reduction="none")+nn.functional.cross_entropy(end,targets[:,1],reduction="none")
    weights=torch.where(targets[:,0]==0,torch.full_like(loss,NEGATIVE_WINDOW_WEIGHT),torch.ones_like(loss))
    return (loss*weights).sum()/weights.sum()

def best_spans(start,end,batch):
    n=start.size(1);indices=torch.arange(n,device=start.device)
    allowed=(indices[None,:]>=indices[:,None])&(indices[None,:]-indices[:,None]<MAX_ANSWER_TOKENS)&(indices[:,None]>=1)
    allowed=allowed[None]&batch["context_mask"][:,:,None]&batch["context_mask"][:,None,:]
    joint=(start[:,:,None]+end[:,None,:]).masked_fill(~allowed,-torch.inf)
    scores,flat=joint.flatten(1).max(-1)
    margin=scores-start[:,0]-end[:,0]
    return (flat//n).tolist(),(flat%n).tolist(),margin.tolist()

@torch.no_grad()
def evaluate(model,loader,return_predictions=False):
    model.eval();best={}
    for cpu_batch in loader:
        batch=to_device(cpu_batch);start,end=model(batch)
        starts,ends,scores=best_spans(start,end,batch)
        for item,a,b,score in zip(batch["items"],starts,ends,scores):
            record=item["record"];qid=record["id"]
            s=item["offsets"][a][0];e=item["offsets"][b][1]
            if qid not in best or score>best[qid][0]:best[qid]=(score,record["context"][s:e],s,e)
    if set(best)!={r["id"] for r in loader.dataset.records}:raise ValueError("Mất question khi gộp windows; không chấm theo feature.")
    totals=np.zeros(2);predictions=[]
    for record in loader.dataset.records:
        score,text,s,e=best[record["id"]]
        refs=[a["text"] for a in record["answers"]]
        totals+=answer_metrics(text,refs)
        if return_predictions:predictions.append({"id":record["id"],"prediction":text,"start":s,"end":e,"references":refs})
    metrics={"exact_match":100*float(totals[0]/len(best)),"token_f1":100*float(totals[1]/len(best)),"question_count":len(best)}
    return (metrics,predictions) if return_predictions else metrics


## 7. Huấn luyện/checkpoint

AdamW và clipping; early stopping primary EM validation tăng, không tune theo token F1 hoặc hidden test. Lưu đủ model/tokenizer/masks/protocol để tạo model mới và tải lại.


In [ ]:
model=build_model(MODEL_CONFIG).to(DEVICE)
print(f"Tham số nhiệm vụ học từ đầu: {sum(p.numel() for p in model.parameters()):,}")
optimizer=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,betas=(0.9,0.98),eps=1e-9,weight_decay=WEIGHT_DECAY)
total_steps=max(1,EPOCHS*len(train_loader))
def lr_multiplier(step):
    step+=1
    return min(step/max(1,WARMUP_STEPS),math.sqrt(max(1,WARMUP_STEPS)/max(step,WARMUP_STEPS))) if WARMUP_STEPS else 1.0
scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lr_multiplier)
best_score=-math.inf;stale_epochs=0
for epoch in range(1,EPOCHS+1):
    model.train();total=0.;seen=0
    for cpu_batch in tqdm(train_loader,desc="Huấn luyện",leave=False):
        batch=to_device(cpu_batch)
        optimizer.zero_grad(set_to_none=True)
        loss=compute_loss(model,batch)
        if not torch.isfinite(loss):raise FloatingPointError("Loss không hữu hạn; kiểm tra mask/nhãn và cấu hình cố định.")
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP)
        optimizer.step();scheduler.step()
        n=len(batch["ids"]);total+=loss.item()*n;seen+=n
    if not seen:raise ValueError("Train rỗng.")
    loss = total/seen
    metrics=evaluate(model,val_loader)
    score=float(metrics[PRIMARY_METRIC])
    if not math.isfinite(score):raise ValueError("Primary validation metric không hữu hạn.")
    print(f"Epoch{epoch:02d}: loss={loss:.4f}; validation={metrics}")
    if score>best_score:
        best_score=score;stale_epochs=0
        torch.save({
            "model_state": model.state_dict(),
            "tokenizer": TOKENIZER_STATE,
            "epoch": epoch,
        }, CHECKPOINT_PATH)
        print("Đã lưu checkpoint tốt nhất theo primary metric validation tăng.")
    else:
        stale_epochs+=1
        if stale_epochs>=PATIENCE:
            print(f"Dừng sớm sau {PATIENCE} epoch không cải thiện.");break


## 8. Reload và held-out official validation

Đánh giá sau reload trên original validation có nhãn. Đây không phải kết quả official hidden test. Không chấm riêng từng window hay chỉ windows dương.


In [ ]:
checkpoint=torch.load(CHECKPOINT_PATH,map_location=DEVICE,weights_only=True)
restore_tokenizer(checkpoint["tokenizer"])
# Cấu hình kiến trúc ở notebook; kích thước vocab được lấy sau khi restore.
inference_config = {"n_words":len(word_vocab),"n_chars":len(char_vocab),"hidden_dim":HIDDEN_DIM,"dropout":DROPOUT}
best_model=build_model(inference_config).to(DEVICE)
best_model.load_state_dict(checkpoint["model_state"])
best_model.eval()
validation_metrics=evaluate(best_model,val_loader)
test_metrics,predictions=evaluate(best_model,test_loader,return_predictions=True)
results={"source":SOURCE,"epoch":checkpoint["epoch"],"validation":validation_metrics,
         "held_out":test_metrics,"split_fingerprints":SPLIT_FINGERPRINTS}
(ARTIFACT_DIR/"metrics.json").write_text(json.dumps(results,ensure_ascii=False,indent=2),encoding="utf-8")
(ARTIFACT_DIR/"predictions.json").write_text(json.dumps(predictions,ensure_ascii=False,indent=2),encoding="utf-8")
print("Đã nạp checkpoint epoch",checkpoint["epoch"])
print("Validation:",validation_metrics)
print("Held-out:",test_metrics)
print("Ví dụ:",predictions[0] if predictions else "Không có mẫu")


## 9. Suy luận câu hỏi mới

Chỉ nhận context và question, không cần answer_start/reference. Trả substring từ context nguyên bản, không sinh văn bản ngoài context.


In [ ]:
@torch.no_grad()
def answer_question(context,question):
    if not context.strip() or not question.strip():raise ValueError("Cần context/question không rỗng.")
    record={"id":"new-question","title":"new","context":context,"question":question,"answers":[]}
    dataset=QADataset([record]);best=None
    best_model.eval()
    for cpu_batch in make_loader(dataset):
        batch=to_device(cpu_batch);start,end=best_model(batch)
        starts,ends,scores=best_spans(start,end,batch)
        for item,a,b,score in zip(batch["items"],starts,ends,scores):
            s=item["offsets"][a][0];e=item["offsets"][b][1]
            if best is None or score>best[0]:best=(score,s,e)
    return {"text":context[best[1]:best[2]],"start":best[1],"end":best[2]}

sample=test_records[0]
print(answer_question(sample["context"],sample["question"]))
